# Image Classification using ResNet

ResNet (Residual Network) is a deep neural network architecture introduced by Microsoft Research in 2015. The key insight behind it is the **residual connection** — a shortcut that lets the output of one layer skip ahead and be added directly to the output of a deeper layer. This solves the vanishing gradient problem that plagued very deep networks before ResNet: gradients could now flow back through the shortcut path without shrinking to near-zero, making it actually possible to train networks with hundreds of layers.

For image classification, ResNet brings a few concrete advantages:

- **Enables deeper networks** — Residual connections make it feasible to train networks with 50, 101, or even 152 layers without degradation in performance.
- **Improved performance** — Deeper networks capture more abstract and nuanced features, which translates into better accuracy on complex datasets.
- **Better generalization** — The skip connections act as a form of implicit regularization, helping the model avoid overfitting and perform well on unseen data.

## Step 1 — Import Libraries

In [1]:
import tensorflow as tf                                              # core deep learning framework
from tensorflow.keras.applications import ResNet50                   # pre-trained ResNet50 model
from tensorflow.keras.datasets import cifar10                        # CIFAR-10 dataset loader
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # image augmentation utility (imported for potential future use)
from tensorflow.keras.models import Sequential                       # sequential model API for stacking layers
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D    # dense output layer and global pooling layer
from tensorflow.keras.optimizers import Adam                         # Adam optimizer with configurable learning rate
from tensorflow.keras.utils import to_categorical                    # converts integer labels to one-hot vectors

## Step 2 — Load and Preprocess CIFAR-10

CIFAR-10 comes in with pixel values in the range [0, 255], so the first thing I do is cast everything to float and divide by 255 to bring it into [0, 1]. Networks train much more stably with small inputs — large pixel values can push activations into saturated regions and slow down learning. The labels come as integers (0–9), so I one-hot encode them to match the 10-unit softmax output layer.

In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()  # load CIFAR-10 split into train and test sets
x_train = x_train.astype('float32') / 255.0                 # cast to float32 and normalize training images to [0, 1]
x_test = x_test.astype('float32') / 255.0                   # same normalization for test images
y_train = to_categorical(y_train, 10)                        # one-hot encode training labels across 10 classes
y_test = to_categorical(y_test, 10)                          # one-hot encode test labels across 10 classes

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


## Step 3 — Load ResNet50 Pre-trained on ImageNet

Rather than training ResNet50 from scratch — which would need a lot more data and compute — I'm loading weights pre-trained on ImageNet. The `include_top=False` flag removes the original 1000-class classification head so I can attach my own layers for CIFAR-10. Setting `base_model.trainable = False` freezes all the ResNet weights, meaning only my new layers will update during training. This is the standard transfer learning setup: reuse the feature extractor, train only the classifier on top.

In [3]:
base_model = ResNet50(weights='imagenet',        # load weights pre-trained on the ImageNet dataset
                      include_top=False,         # exclude the original fully connected classification head
                      input_shape=(32, 32, 3))   # set input shape to match CIFAR-10 images (32x32 RGB)
base_model.trainable = False                     # freeze all ResNet layers so only the new top layers are trained

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Step 4 — Build the Classification Model

`GlobalAveragePooling2D` takes the feature maps output by the frozen ResNet base and computes the average across each spatial location, producing a compact 1D vector. This is a cleaner alternative to flattening because it drastically reduces the number of parameters and is less prone to overfitting. From there, a 1024-unit dense layer with ReLU learns task-specific combinations of those features, and the final Softmax layer maps everything down to 10 class probabilities.

In [4]:
model = Sequential([
    base_model,                              # frozen ResNet50 backbone for feature extraction
    GlobalAveragePooling2D(),                # average each feature map spatially to produce a 1D feature vector
    Dense(1024, activation='relu'),          # fully connected layer to learn CIFAR-10-specific feature combinations
    Dense(10, activation='softmax')          # output layer: one probability per CIFAR-10 class
])
model.summary()                             # print a full breakdown of layers and parameter counts

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 1, 1, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        10,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,696,138 (98.02 MB)

 Trainable params: 2,108,426 (8.04 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

## Step 5 — Compile the Model

I'm using Adam with a lower-than-default learning rate of `0.0001`. Since only the top layers are being trained (the ResNet base is frozen), a smaller learning rate helps avoid large weight updates that could push the new layers in the wrong direction early on. Categorical crossentropy is the right loss here because we're doing multi-class classification with one-hot encoded targets — it measures the difference between the predicted probability distribution and the true one.

In [5]:
model.compile(optimizer=Adam(learning_rate=0.0001),  # Adam with a small learning rate to gently train the new top layers
              loss='categorical_crossentropy',        # standard loss for multi-class classification with one-hot labels
              metrics=['accuracy'])                   # track accuracy so we can monitor progress during training

## Step 6 — Train the Model

In [6]:
model.fit(x_train, y_train,               # train on the full normalized training set
          batch_size=64,                  # process 64 images per gradient update
          epochs=10,                      # run 10 full passes over the training data
          validation_data=(x_test, y_test))  # evaluate on the test set at the end of each epoch

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 31s 24ms/step - accuracy: 0.2360 - loss: 2.0986 - val_accuracy: 0.2524 - val_loss: 2.0081
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 11ms/step - accuracy: 0.3014 - loss: 1.9374 - val_accuracy: 0.3278 - val_loss: 1.8861
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.3280 - loss: 1.8781 - val_accuracy: 0.3380 - val_loss: 1.8595
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.3433 - loss: 1.8374 - val_accuracy: 0.3540 - val_loss: 1.8084
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.3551 - loss: 1.8065 - val_accuracy: 0.3704 - val_loss: 1.7885
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.3638 - loss: 1.7822 - val_accuracy: 0.3666 - val_loss: 1.7659
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.3727 - loss: 1.7608 - val_accuracy: 0.3591 - val_loss: 1.7775
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.3788 - loss: 1.7428 - val_ac

## Step 7 — Evaluate the Model

In [7]:
test_loss, test_acc = model.evaluate(x_test, y_test)  # run a forward pass over the full test set to get loss and accuracy
print(f"Test accuracy: {test_acc}")                    # print the final test accuracy

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.3856 - loss: 1.7227
Test accuracy: 0.3856000006198883
